<a href="https://colab.research.google.com/github/Deepshika-Mekala/deepshikamekala.github.io/blob/main/Deepshika_Mekala_data%26Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import numpy as np

reading sales_builder_a & sales_builder_b files

In [ ]:
sales_a = pd.read_csv("sales_builder_a.csv")
sales_b = pd.read_csv("sales_builder_b.csv")

sales_a["builder"] = "a"
sales_b["builder"] = "b"

sales = pd.concat([sales_a, sales_b], ignore_index=True)

print(sales.columns.tolist())
sales.head()


['home_id', 'sale_contract_date', 'community_name', 'sale_type', 'total_sale_price', 'sqft', 'builder']


,home_id,sale_contract_date,community_name,sale_type,total_sale_price,sqft,builder
0,HS-FVE-000247,2025-03-12,Fairview Estates,spec,388737.0,2068,a
1,HS-FVE-000115,2024-06-06,Fairview Estates,spec,372636.0,2152,a
2,HS-GLN-000108,2024-05-27,Glenview Meadows,spec,541610.0,3116,a
3,HS-FVE-000319,2025-08-28,Fairview Estates,spec,314469.0,2273,a
4,HS-RIV-000169,2024-10-04,Riverbend Townhomes,build,331726.0,1460,a


preparing sales data

In [ ]:
sales = sales.rename(columns={
    "sale_contract_date": "date",
    "community_name": "community"
})

sales["date"] = pd.to_datetime(sales["date"], errors="coerce")
sales["month"] = sales["date"].dt.to_period("M").dt.to_timestamp()

sales["actual_sales"] = 1

sales_monthly = (
    sales.groupby(["builder", "community", "month"], as_index=False)
         .agg(actual_sales=("actual_sales", "sum"))
)

sales_monthly.head()

,builder,community,month,actual_sales
0,a,Cedar Creek,2024-01-01,1
1,a,Cedar Creek,2024-02-01,3
2,a,Cedar Creek,2024-03-01,2
3,a,Cedar Creek,2024-05-01,2
4,a,Cedar Creek,2024-06-01,2


for target_sales_builder_a

In [ ]:
targets_raw = pd.read_excel("target_sales_builder_a.xlsx", header=None)
targets_raw.head(6)


,0
0,;;;;;2025;2025;2025;2025;2025;2025;2025;2025;2...
1,;;;;;Projected;Projected;Projected;Projected;P...
2,Company;Department;Sub Code;Subdivision;;Jan-2...
3,003;11;003-HS-FVE;Fairview Estates;Sales;7;7;7...
4,003;11;003-HS-RIV;Riverbend Townhomes;Sales;4;...
5,003;11;003-HS-GLN;Glenview Meadows;Sales;2;1;2...


In [ ]:
targets_split = targets_raw[0].astype(str).str.split(";", expand=True)#everything was in one column separated by semicolons

targets_split.columns = targets_split.iloc[2]
targets_split = targets_split.iloc[3:].reset_index(drop=True)


targets_split.columns = targets_split.columns.astype(str).str.strip()


subdivision_candidates = [c for c in targets_split.columns if "sub" in c.lower()]

if not subdivision_candidates:
    raise ValueError(f"Couldn't find Subdivision column. Columns are: {targets_split.columns.tolist()}")

sub_col = subdivision_candidates[0]  # take the first match


targets_a = targets_split.rename(columns={sub_col: "community"}).copy()
targets_a["builder"] = "a"
targets_a["community"] = targets_a["community"].astype(str).str.strip()


month_cols = [c for c in targets_a.columns if "-" in c and len(c) == 6]  # basic filter for "Mon-YY"
if not month_cols:

    meta_cols = {"Company", "Department", "Sub Code", "community", "builder"}
    month_cols = [c for c in targets_a.columns if c not in meta_cols]


targets_a_long = targets_a.melt(
    id_vars=["builder", "community"],
    value_vars=month_cols,
    var_name="month",
    value_name="target_sales"
)

targets_a_long["month"] = pd.to_datetime(targets_a_long["month"], format="%b-%y", errors="coerce")
targets_a_long["month"] = targets_a_long["month"].dt.to_period("M").dt.to_timestamp()
targets_a_long["target_sales"] = pd.to_numeric(targets_a_long["target_sales"], errors="coerce")


print(targets_a_long.head())



  builder   community      month  target_sales
0       a  003-HS-FVE 2025-01-01             7
1       a  003-HS-RIV 2025-01-01             4
2       a  003-HS-GLN 2025-01-01             2
3       a  003-HS-CCR 2025-01-01             1
4       a  003-HS-FVE 2025-02-01             7


for target_sales_builder_b

In [ ]:
targets_b = pd.read_csv("target_sales_builder_b.csv")
targets_b["builder"] = "b"
targets_b["community"] = targets_b["community"].astype(str).str.strip()

targets_b["month"] = pd.to_datetime(
    targets_b["year"].astype(str) + "-" + targets_b["month"].astype(str) + "-01",
    errors="coerce"
)

targets_b_long = targets_b.rename(columns={"sales_target": "target_sales"})[
    ["builder", "community", "month", "target_sales"]
]
targets_b_long["target_sales"] = pd.to_numeric(targets_b_long["target_sales"], errors="coerce")

targets_b_long.head()


,builder,community,month,target_sales
0,b,Maplewood Heights,2025-01-01,0
1,b,Maplewood Heights,2025-02-01,0
2,b,Maplewood Heights,2025-03-01,0
3,b,Maplewood Heights,2025-04-01,0
4,b,Maplewood Heights,2025-05-01,0


In [ ]:
targets_all = pd.concat([targets_a_long, targets_b_long], ignore_index=True)

targets_monthly = (
    targets_all.groupby(["builder", "community", "month"], as_index=False)
               .agg(target_sales=("target_sales", "sum"))
)

report = sales_monthly.merge(
    targets_monthly,
    on=["builder", "community", "month"],
    how="outer"
)

report["actual_sales"] = report["actual_sales"].fillna(0).astype(int)
report["target_sales"] = report["target_sales"].fillna(0)

report["variance"] = report["actual_sales"] - report["target_sales"]

report.head(20)


,builder,community,month,actual_sales,target_sales,variance
0,a,Cedar Creek,2024-01-01,1,0.0,1.0
1,a,Cedar Creek,2024-02-01,3,0.0,3.0
2,a,Cedar Creek,2024-03-01,2,0.0,2.0
3,a,Cedar Creek,2024-05-01,2,0.0,2.0
4,a,Cedar Creek,2024-06-01,2,0.0,2.0
5,a,Cedar Creek,2024-07-01,3,0.0,3.0
6,a,Cedar Creek,2024-11-01,5,0.0,5.0
7,a,Cedar Creek,2024-12-01,2,0.0,2.0
8,a,Cedar Creek,2025-01-01,1,0.0,1.0
9,a,Cedar Creek,2025-06-01,2,0.0,2.0


Fetch CRM Leads and Web Traffic Data

In [ ]:
API_KEY = "bapi_sk_c8f9a2b7e4d1c5a3f6b9d2e7a1c4b8f5"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

CRM_URL = "https://builder-api-875175326233.us-central1.run.app/builder-api/crm"
TRAFFIC_URL = "https://builder-api-875175326233.us-central1.run.app/builder-api/web-traffic"

def fetch_api(url, builder):
    r = requests.get(url, headers=HEADERS, params={"builder": builder}, timeout=30)
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    return pd.DataFrame(data)

crm = pd.concat([fetch_api(CRM_URL, "a").assign(builder="a"),
                 fetch_api(CRM_URL, "b").assign(builder="b")], ignore_index=True)

traffic = pd.concat([fetch_api(TRAFFIC_URL, "a").assign(builder="a"),
                     fetch_api(TRAFFIC_URL, "b").assign(builder="b")], ignore_index=True)

print("CRM columns:", crm.columns.tolist())
print("Traffic columns:", traffic.columns.tolist())

crm.head()


CRM columns: ['hs_object_id', 'createdate', 'community', 'hs_analytics_source', 'hs_lead_status', 'builder', 'registantid', 'create_at', 'sourcetype', 'rating']
Traffic columns: ['date', 'community', 'sessions', 'users', 'builder']


,hs_object_id,createdate,community,hs_analytics_source,hs_lead_status,builder,registantid,create_at,sourcetype,rating
0,18444005069,2025-09-21T08:38:53Z,Riverbend Townhomes,FRIEND_REFERRAL,OPEN,a,NaN,NaN,NaN,NaN
1,18444001688,2024-07-30T20:49:22Z,Glenview Meadows,SOCIAL_MEDIA,OPEN,a,NaN,NaN,NaN,NaN
2,18444002314,2024-10-14T12:18:18Z,Riverbend Townhomes,REFERRALS,NEW_LEAD,a,NaN,NaN,NaN,NaN
3,18444002977,2025-01-29T04:24:39Z,Fairview Estates,PAID_SEARCH,NEW_LEAD,a,NaN,NaN,NaN,NaN
4,18444005839,2025-12-12T15:58:41Z,Riverbend Townhomes,PAID_SEARCH,OPEN,a,NaN,NaN,NaN,NaN


Preparing Monthly CRM  Metrics

In [ ]:
crm["createdate"] = pd.to_datetime(crm["createdate"], errors="coerce")
crm["month"] = crm["createdate"].dt.to_period("M").dt.to_timestamp()

crm_monthly = (
    crm.groupby(["builder", "community", "month"], as_index=False)
       .size()
       .rename(columns={"size": "crm_lead_count"})
)

crm_monthly.head()


C:\Users\deeps\AppData\Local\Temp\ipykernel_20740\4255153834.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  crm["month"] = crm["createdate"].dt.to_period("M").dt.to_timestamp()


,builder,community,month,crm_lead_count
0,a,Cedar Creek,2024-01-01,21
1,a,Cedar Creek,2024-02-01,16
2,a,Cedar Creek,2024-03-01,16
3,a,Cedar Creek,2024-04-01,21
4,a,Cedar Creek,2024-05-01,25


Preparing Monthly Traffic Metrics

In [ ]:

traffic["date"] = pd.to_datetime(traffic["date"], errors="coerce")
traffic["month"] = traffic["date"].dt.to_period("M").dt.to_timestamp()

traffic_monthly = (
    traffic.groupby(["builder", "community", "month"], as_index=False)
           .agg(web_traffic=("sessions", "sum"))
)

traffic_monthly.head()


,builder,community,month,web_traffic
0,a,Cedar Creek at Pine Run,2024-01-01,927
1,a,Cedar Creek at Pine Run,2024-02-01,824
2,a,Cedar Creek at Pine Run,2024-03-01,1041
3,a,Cedar Creek at Pine Run,2024-04-01,828
4,a,Cedar Creek at Pine Run,2024-05-01,971


Combining Sales, Targets, CRM, and Web Traffic for final sheet

In [ ]:
final_df = (
    report
    .merge(crm_monthly, on=["builder", "community", "month"], how="left")
    .merge(traffic_monthly, on=["builder", "community", "month"], how="left")
)

final_df["crm_lead_count"] = final_df["crm_lead_count"].fillna(0).astype(int)
final_df["web_traffic"] = final_df["web_traffic"].fillna(0).astype(int)

final_df["lead_to_sale_rate"] = final_df.apply(
    lambda x: x["actual_sales"] / x["crm_lead_count"] if x["crm_lead_count"] > 0 else None,
    axis=1
)

final_df.head(20)


,builder,community,month,actual_sales,target_sales,variance,crm_lead_count,web_traffic,lead_to_sale_rate
0,a,Cedar Creek,2024-01-01,1,0.0,1.0,21,0,0.047619
1,a,Cedar Creek,2024-02-01,3,0.0,3.0,16,0,0.187500
2,a,Cedar Creek,2024-03-01,2,0.0,2.0,16,0,0.125000
3,a,Cedar Creek,2024-05-01,2,0.0,2.0,25,0,0.080000
4,a,Cedar Creek,2024-06-01,2,0.0,2.0,19,0,0.105263
5,a,Cedar Creek,2024-07-01,3,0.0,3.0,15,0,0.200000
6,a,Cedar Creek,2024-11-01,5,0.0,5.0,16,0,0.312500
7,a,Cedar Creek,2024-12-01,2,0.0,2.0,13,0,0.153846
8,a,Cedar Creek,2025-01-01,1,0.0,1.0,19,0,0.052632
9,a,Cedar Creek,2025-06-01,2,0.0,2.0,15,0,0.133333


Note: Web traffic returned as 0 for many community-months in the API response, so I did not calculate traffic-to-lead conversion to avoid misleading ratios.


In [ ]:
final_df.to_csv("final_monthly_community_report.csv", index=False)

